# Hybrid Quantum-Classical Reinforcement Learning (2/3)

___
___

## Introduction
___

This series of experiments is based on the 2025 paper by Nagy et al.: "[Hybrid Quantum-Classical Reinforcement Learning in Latent Observation Spaces](https://arxiv.org/abs/2410.18284)".

In this article, the authors apply hybrid quantum-classical Reinforcement Learning (RL) models to two simulated environments. They compare classical, qubit-based and photonic-based agents, all using Proximal Policy Optimization (PPO). They also use an AutoEncoder (AE) to compress the dimensionality of the observations and train that AE jointly with the PPO agents.

The notebook series aims to compare the resources cost of the different systems to reach the same performance. It is divided in three parts:

- [Part I: Classical vs Qubit agents on the Cart Pole environment](QRL_experiment_1.ipynb)
- **Part II: Classical vs Qubit agents on the Lunar Lander and Car Racing environments**
- [Part III: Photonic agents on the three environments](QRL_experiment_3.ipynb)

This second notebook will present:
1. The Lunar Lander environment
2. The Car Racing environment
3. Review of previous notebook's code
4. Results
5. Conclusions
6. Follow-up

In [ ]:
# !pip install ipynb torch torchvision swig gymnasium[box2d] matplotlib pennylane
import datetime
import random
import resource
from collections import namedtuple, deque
from itertools import count
import numpy as np
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import pennylane as qp

from ipynb.fs.defs.QRL_experiment_1 import EncoderDecoderNN, AutoEncoder, PPO, ActorCriticNN, ActorQubitNN

# For reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

## 1 - The Lunar Lander environment
___

In this Box2D environment, the agent learns to land a spaceship in a designated area. 

<div align="center">
<img src="images/qrl_demo_lunarlander.gif" width="300"/>
</div>

At each step, the agent gets 8 observations: the x and y coordinates of the ship, its x and y linear velocities, its angle, angular velocity, and whether each leg of the ship is touching the ground. The agent can choose between 4 actions: do nothing, fire the main engine, fire the left engine or fire the right one.

A reward is granted at each step, depending on the ship's position compared to the landing pad, its velocity, orientation, engine firings and contact with the ground. An additional reward is given at the end depending on whether the ship landed correctly or not. The episode ends when the ship crashes, is getting out of display or stalls.

More information about this environment can be found here: https://gymnasium.farama.org/environments/box2d/lunar_lander/.

In [ ]:
lunarlander_env = gym.make("LunarLander-v3", render_mode="rgb_array")
lunarlander_env.reset(seed=seed)
lunarlander_env.action_space.seed(seed)
lunarlander_env.observation_space.seed(seed)

## 2 - The Car Racing environment
___

In this Box2D environment, the agent learns to drive a car on a circuit. 

<div align="center">
<img src="images/qrl_demo_carracing.gif" width="300"/>
</div>

At each step, the agent gets a 96x96 RGB image as observations. It can then choose between 5 actions: do nothing, steer right, steer left,  gas or brake.

The reward is -0.1 at every step and +1000/N for every track tile visited, where N is the total number of tiles visited in the track. For example, if you have finished in 732 frames, your reward is 1000 - 0.1*732 = 926.8 points. The episode finishes when all the tiles are visited. The car can also go outside the playfield - that is, far off the track, in which case it will receive -100 reward and die.

More information about this environment can be found here: https://gymnasium.farama.org/environments/box2d/car_racing/.

In [ ]:
carracing_env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
carracing_env.reset(seed=seed)
carracing_env.action_space.seed(seed)
carracing_env.observation_space.seed(seed)

## 3 - Reused code from the first notebook
___

The first notebook introduced AutoEncoders, as well as classical and qubit Proximal Policy Optimization (PPO) agents.

For the Car Racing env however, since the input is an image, we will use an AutoEncoder and Critic model with Convolutional layers.

Convolutions

In [ ]:
class EncoderCNN(nn.Module):
    def __init__(self, input_shape, hidden_dims, output_dim, kernel_size=5, stride_size=2, pooling_size=2):
        super().__init__()
        width, _, input_channels = input_shape
        self.conv1 = nn.Conv2d(input_channels, hidden_dims[0], kernel_size, stride_size)
        self.pool1 = nn.MaxPool2d(pooling_size)
        conv1_size = int((width-kernel_size)/stride_size+1) // pooling_size
        self.conv2 = nn.Conv2d(hidden_dims[0], hidden_dims[1], kernel_size, stride_size)
        self.pool2 = nn.MaxPool2d(pooling_size)
        conv2_size = int((conv1_size-kernel_size)/stride_size+1) // pooling_size
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(hidden_dims[1] * conv2_size**2, output_dim)
        # self.fc1 = nn.Linear(hidden_dims[1] * conv2_size**2, hidden_dims[2])
        # self.fc2 = nn.Linear(hidden_dims[2], output_dim)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        return self.fc1(x)
        # x = F.relu(self.fc1(x))
        # return self.fc2(x)

class DecoderCNN(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_shape, kernel_size=5, stride_size=2, pooling_size=2):
        super().__init__()
        width, _, output_channels = output_shape
        self.conv1 = nn.ConvTranspose2d(hidden_dims[0], output_channels, kernel_size, stride_size, output_padding=1)
        self.pool1 = nn.Upsample(scale_factor=pooling_size)
        conv1_size = int((width-kernel_size)/stride_size+1) // pooling_size
        self.conv2 = nn.ConvTranspose2d(hidden_dims[1], hidden_dims[0], kernel_size, stride_size)
        self.pool2 = nn.Upsample(scale_factor=pooling_size)
        conv2_size = int((conv1_size-kernel_size)/stride_size+1) // pooling_size
        self.intermediate_shape = (-1, hidden_dims[1], conv2_size, conv2_size)
        self.fc1 = nn.Linear(input_dim, hidden_dims[1] * conv2_size**2)
        # self.fc1 = nn.Linear(hidden_dims[2], hidden_dims[1] * conv2_size**2)
        # self.fc2 = nn.Linear(input_dim, hidden_dims[2])
        
    def forward(self, x):
        # x = F.relu(self.fc2(x))
        x = self.fc1(x)
        x = x.view(self.intermediate_shape)
        x = self.pool2(x)
        x = F.relu(self.conv2(x))
        x = self.pool1(x)
        x = F.sigmoid(self.conv1(x))
        return x.permute(0, 2, 3, 1)

class ConvolutionalAE(nn.Module):
    def __init__(self, input_shape, parameters):
        super().__init__()
        self.encoder = EncoderCNN(input_shape, parameters["ae_hidden_dims"], parameters["ae_output_dim"])
        self.decoder = DecoderCNN(parameters["ae_output_dim"], parameters["ae_hidden_dims"], input_shape)

In [ ]:
class CriticCNN(nn.Module):
    ''' A simple Convolutional Neural Network that will be the model basis for the Critic '''
    def __init__(self, input_shape, intermediate_dim, output_dim, kernel_size=5, stride_size=2, pooling_size=2):
        super().__init__()
        width, _, input_channels = input_shape
        self.conv1 = nn.Conv2d(input_channels, intermediate_dim[0], kernel_size, stride_size)
        self.pool1 = nn.MaxPool2d(pooling_size)
        conv1_size = int((width-kernel_size)/stride_size+1) // pooling_size
        self.conv2 = nn.Conv2d(intermediate_dim[0], intermediate_dim[1], kernel_size, stride_size)
        self.pool2 = nn.MaxPool2d(pooling_size)
        conv2_size = int((conv1_size-kernel_size)/stride_size+1) // pooling_size
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(intermediate_dim[1] * conv2_size**2, output_dim)
        # self.fc1 = nn.Linear(intermediate_dim[1] * conv2_size**2, intermediate_dim[2])
        # self.fc2 = nn.Linear(intermediate_dim[2], output_dim)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        return self.fc1(x)
        # x = F.relu(self.fc1(x))
        # return self.fc2(x)

## 4 - Results
___

### Lunar Lander

In [ ]:
ll_config = {
    "ae_hidden_dim": 64, "ae_output_dim": 3, 
    "critic_intermediate_dim": 64,
    "K": 40, "episode_update_frequency": 2, "mean_reward_lookback": 50, "mean_reward_stop": 100,
    "gamma": 0.99, "epsilon": 0.2, "entropy_coeff": 0.01
    }

In [ ]:
# 1. Classical PPO
ll_config["actor_intermediate_dim"] = 64
ll_classical_ppo = PPO(AutoEncoderClass=AutoEncoder, ActorClass=ActorCriticNN, CriticClass=ActorCriticNN, env=lunarlander_env, config=ll_config)
ll_classical_mean_rewards = ll_classical_ppo.run()

In [ ]:
ll_classical_ppo.evaluate()

In [ ]:
# 2. Qubit PPO
ll_config["actor_intermediate_dim"] = 4
ll_qubit_ppo = PPO(AutoEncoderClass=AutoEncoder, ActorClass=ActorQubitNN, CriticClass=ActorCriticNN, env=lunarlander_env, config=ll_config)
ll_qubit_mean_rewards = ll_qubit_ppo.run()

In [ ]:
ll_qubit_ppo.evaluate()

### Car Racing

In [ ]:
cr_config = {
    "ae_hidden_dims": [8, 8], "ae_output_dim": 6,
    "critic_intermediate_dim": [8, 8],
    "K": 40, "episode_update_frequency": 2, "mean_reward_lookback": 50, "mean_reward_stop": 800,
    "gamma": 0.99, "epsilon": 0.2, "entropy_coeff": 0.01
    }

In [ ]:
# 1. Classical PPO
cr_config["actor_intermediate_dim"] = 64
cr_classical_ppo = PPO(AutoEncoderClass=ConvolutionalAE, ActorClass=ActorCriticNN, CriticClass=CriticCNN, env=carracing_env, config=cr_config)
cr_classical_mean_rewards = cr_classical_ppo.run()

In [ ]:
cr_classical_ppo.evaluate()

In [ ]:
# 2. Qubit PPO
cr_config["actor_intermediate_dim"] = 5
cr_qubit_ppo = PPO(AutoEncoderClass=ConvolutionalAE, ActorClass=ActorQubitNN, CriticClass=CriticCNN, env=carracing_env, config=cr_config)
cr_qubit_mean_rewards = cr_qubit_ppo.run()

In [ ]:
cr_qubit_ppo.evaluate()

## 5 - Conclusions
___

Bigger environments -> Time constraint but how do performances, memory and params follow? For both classical and qubits?

In [ ]:
plt.figure()
plt.axhline(y=ll_config["mean_reward_stop"], color="black")
plt.plot(ll_classical_mean_rewards, label="Classical PPO")
plt.plot(ll_qubit_mean_rewards, label="Qubit-based PPO")
plt.xlabel("Episodes")
plt.ylabel("Rewards Mean Average")
plt.title("Performance")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.axhline(y=cr_config["mean_reward_stop"], color="black")
plt.plot(cr_classical_mean_rewards, label="Classical PPO")
plt.plot(cr_qubit_mean_rewards, label="Qubit-based PPO")
plt.xlabel("Episodes")
plt.ylabel("Rewards Mean Average")
plt.title("Performance")
plt.legend()
plt.show()

## 6 - Follow-up
___

Again, Try using other values for actor_intermediate_dim, max stop and lookback

In the next notebook, the three environments seen in the series (Cart Pole, Lunar Lander and Car Racing) will be tested with a Photonic PPO.